# GTSRB Traffic Sign Classification — CNN

정리된 실행용 노트북입니다. 원본 실험의 **보고된 성능(테스트 91.58%)을 새로 재현한 것은 아닙니다.** 데이터 경로 설정 후 전체 셀을 실행하여 새 결과를 확인하세요.

## 1. 환경 설정 및 데이터 경로

[Kaggle GTSRB](https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign) 데이터를 내려받아 압축을 해제합니다. 데이터 루트에 `Train/`과 `Test.csv`가 있어야 합니다. 환경 변수 `GTSRB_DATA_DIR` 또는 `DATA_ROOT`를 수정하세요.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = Path(os.environ.get("GTSRB_DATA_DIR", "./data")).expanduser().resolve()
TRAIN_DIR = DATA_ROOT / "Train"
TEST_CSV = DATA_ROOT / "Test.csv"
assert TRAIN_DIR.is_dir(), f"Train folder missing: {TRAIN_DIR}"
assert TEST_CSV.exists(), f"Test.csv missing: {TEST_CSV}"
print("Dataset root:", DATA_ROOT)
print("TensorFlow:", tf.__version__)

## 2. 클래스 순서 확인 및 데이터셋 구성

**중요:** 파일 디렉터리의 클래스명이 문자열로 정렬됩니다. 따라서 학습 데이터에서 생성한 `class_names`를 테스트 제너레이터에도 전달해야 합니다.

In [ ]:
IMG_SIZE = (32, 32)
BATCH_SIZE = 128

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="training",
    seed=SEED, image_size=IMG_SIZE, label_mode="categorical",
    batch_size=BATCH_SIZE, shuffle=True,
)
valid_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="validation",
    seed=SEED, image_size=IMG_SIZE, label_mode="categorical",
    batch_size=BATCH_SIZE, shuffle=False,
)
class_names = train_raw.class_names
assert len(class_names) == 43
print("Class order:", class_names[:15])

rescale = tf.keras.layers.Rescaling(1.0 / 255)
train_ds = train_raw.map(lambda x, y: (rescale(x), y)).prefetch(tf.data.AUTOTUNE)
valid_ds = valid_raw.map(lambda x, y: (rescale(x), y)).prefetch(tf.data.AUTOTUNE)

In [ ]:
test_metadata = pd.read_csv(TEST_CSV)
assert {"Path", "ClassId"}.issubset(test_metadata.columns)
test_metadata["ClassId"] = test_metadata["ClassId"].astype(str)
assert set(test_metadata["ClassId"].unique()).issubset(set(class_names))
missing_paths = [p for p in test_metadata["Path"] if not (DATA_ROOT / p).is_file()]
assert not missing_paths, f"Missing {len(missing_paths)} test images; first: {missing_paths[:3]}"

test_generator = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0/255)
test_ds = test_generator.flow_from_dataframe(
    dataframe=test_metadata, directory=str(DATA_ROOT),
    x_col="Path", y_col="ClassId", classes=class_names,
    target_size=IMG_SIZE, class_mode="categorical",
    batch_size=BATCH_SIZE, shuffle=False,
)
assert test_ds.n == len(test_metadata)
assert list(test_ds.class_indices.keys()) == class_names
print("Test samples:", test_ds.n)

## 3. CNN 모델 및 학습

원본 개선 실험과 같은 레이어 구성(Conv32/64, Dense64, Dropout0.2), 학습률 0.001, 10 epochs로 구성합니다. 라이브러리 버전이나 환경에 따라 결과는 다를 수 있습니다.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy", metrics=["accuracy"],
)
model.summary()
history = model.fit(train_ds, epochs=10, validation_data=valid_ds)

## 4. 성능 평가

아래 출력이 **새 실행 결과**입니다. README의 91.58%는 원본 노트북의 기록이며, 여기에 적힌 코드는 실행 검증되지 않았습니다.

In [ ]:
train_accuracy = history.history["accuracy"][-1]
val_accuracy = history.history["val_accuracy"][-1]
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Train accuracy (final epoch): {train_accuracy:.2%}")
print(f"Validation accuracy (final epoch): {val_accuracy:.2%}")
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set(title="Accuracy", xlabel="Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set(title="Loss", xlabel="Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. 추후 분석

클래스별 precision/recall/F1, confusion matrix, 오분류 샘플 시각화는 현재 미실시입니다. 데이터 라이선스와 실험 재현 여부를 확인한 뒤 보완할 예정입니다.